# Arbeitspaket (AP) 1: Management und Nutzung relationale Daten

**Prüfung – 40 Punkte**

### Persönliche Angaben (bitte ergänzen) und Prüfungsregeln

<table>
  <tr><td>Vorname:</td><td></td></tr>
  <tr><td>Nachname:</td><td></td></tr>
  <tr><td>Immatrikulationsnummer:</td><td></td></tr>
  <tr><td>Modul:</td><td>Data Science</td></tr>
  <tr><td>Datum / Zeit:</td><td>09.11.2026 / Prüfungsbeginn 08:00 Uhr</td></tr>
  <tr><td>Erlaubte Hilfsmittel:</td><td>Es gelten die Prüfungsregeln (PDF-Dokument auf EXAM-Moodle).</td></tr>
  <tr><td>Nicht erlaubt:</td><td>Es gelten die Prüfungsregeln (PDF-Dokument auf EXAM-Moodle).</td></tr>
  <tr><td>Technische Probleme:</td><td>Wenden Sie sich bei technischen Problemen umgehend an die Prüfungsaufsicht.</td></tr>
</table>

## Bewertungskriterien

| Teil | Aufgaben | Punkte |
|---|---|---:|
| A: Konzeptfragen mit Datenbezug | A1–A3 | 12 |
| B: Fehleranalyse und Korrektur | B1–B2 | 16 |
| C: SQL-Abfragen schreiben | C1–C3 | 12 |
| **Total** | **8 Aufgaben** | **40** |

### Bewertungsschema

Jede Aufgabe wird für sich und **als Ganzes** bewertet. Ihre Punktzahl steht in der Überschrift.

<table>
  <thead>
    <tr><th>Kategorie</th><th>Beschreibung</th><th>Bewertung</th><th>4 Pkt</th><th>8 Pkt</th></tr>
  </thead>
  <tbody>
    <tr><td>Korrekt</td><td>Vollständig und fachlich einwandfrei: Antwort bzw. Begründung korrekt und nachvollziehbar; bei SQL-Aufgaben liefert die SQL-Abfrage das richtige Ergebnis</td><td>100%</td><td>4</td><td>8</td></tr>
    <tr><td>Minimale Mängel</td><td>Kleine Fehler (z.B. falsche Sortierung, fehlende Rundung) oder Begründung ungenau bzw. unvollständig</td><td>75%</td><td>3</td><td>6</td></tr>
    <tr><td>Mittlere Mängel</td><td>Ein wesentlicher Teil der Aufgabe ist korrekt gelöst. Eine weitere wesentliche Anforderung fehlt oder ist fachlich falsch.</td><td>50%</td><td>2</td><td>4</td></tr>
    <tr><td>Gravierende Mängel</td><td>Problem bzw. Ansatz erkennbar, aber Lösung oder Schlussfolgerung im Kern falsch (im Code z.B. ungeeigneter Lösungsweg wie falscher JOIN-Typ oder fehlende Tabelle)</td><td>25%</td><td>1</td><td>2</td></tr>
    <tr><td>Nicht lauffähig / nicht sinnvoll</td><td>Antwort fehlt, kein Bezug zur Aufgabe, oder Code nicht lauffähig (Syntax- oder Laufzeitfehler) und kein korrekter schriftlicher Teil.</td><td>0%</td><td>0</td><td>0</td></tr>
  </tbody>
</table>

Alternative, fachlich korrekte Lösungswege werden ebenfalls als korrekt gewertet.

## Hinweise zu den Daten

Der fiktive Theaterverbund «BühnenNetz» verkauft Tickets für Theateraufführungen.

**Eine Zeile in `tickets` entspricht genau einem verkauften Ticket.**

Die Daten sind in fünf Tabellen gespeichert:

| Tabelle | Bedeutung |
|---|---|
| `plays` | Theaterstücke |
| `theatres` | Theater |
| `performances` | Aufführungen mit Termin und Basispreis |
| `customers` | Kunden |
| `tickets` | Einzelne Tickets mit Kunde, Sitz und bezahltem Preis |

Ein Theaterstück kann mehrfach aufgeführt werden. Jede Aufführung findet in einem Theater statt. Ein Kunde kann mehrere Tickets kaufen. Rabatte sind möglich. Eine Freikarte ist ein Ticket mit `paid_price = 0`.

`theatre.db` enthält 15 Theaterstücke, 6 Theater, 90 Aufführungen, 505 Kunden und 2'000 Tickets. Die kleinen Theater haben jeweils 24 bis 48 Sitzplätze. Die Daten bilden alle ausgegebenen Tickets für die gespeicherten Aufführungen vom 4. September bis 11. Oktober 2026 ab, einschliesslich Freikarten. Ein Theater und ein Theaterstück haben in diesem Zeitraum keine Aufführung.

Die Sitzplätze sind in vier Reihen (`A` bis `D`) angeordnet. Bei 24 Sitzplätzen gibt es beispielsweise je sechs Plätze pro Reihe (`A01` bis `D06`). Sitzplatznummern werden bei anderen Aufführungen erneut verwendet. Ein Kunde kann mehrere Tickets für eine Aufführung kaufen, beispielsweise für Begleitpersonen.

Die Tabellenbeschreibungen und das ER-Diagramm stehen in `theatre_data.xlsx`.

**Start:** Öffnen Sie das Notebook im Ordner AP01, sodass `theatre.db` im Arbeitsverzeichnis liegt. PostgreSQL wird verbunden und mit den Tabellen aus SQLite befüllt.

## Setup (vorgegeben, nicht bewertet)

Diese Zellen einmal von oben nach unten ausführen. Alle Daten liegen lokal vor.

In [ ]:
# Benötigte Bibliotheken importieren.
import os
import sqlite3
import pandas as pd
from sqlalchemy import create_engine, text

# Aktuelles Arbeitsverzeichnis anzeigen.
print('Working directory:', os.getcwd())

In [ ]:
# Zugangsdaten (vorgegeben)
user = 'pgadmin'
password = 'geheim'
port = '5432'
database = 'postgres'

# Erreichbaren PostgreSQL-Host ermitteln.
host = None
for candidate in ('db', 'localhost'):
    probe = create_engine(
        f'postgresql://{user}:{password}@{candidate}:{port}/{database}',
        connect_args={'connect_timeout': 3},
    )
    try:
        with probe.connect() as connection:
            connection.execute(text('SELECT 1;'))
        host = candidate
        break
    except Exception:
        pass
    finally:
        probe.dispose()

# Datenbankverbindung öffnen und prüfen.
db_connection_url = f'postgresql://{user}:{password}@{host}:{port}/{database}'
engine = create_engine(db_connection_url)
with engine.connect() as connection:
    print('Host:', host)
    print('Datenbank:', connection.execute(text('SELECT CURRENT_DATABASE();')).scalar())

In [ ]:
# Daten laden (vorgegeben)
# Bei Verwendung des SQLite-Backups ist keine Übertragung nötig.
if engine.dialect.name == 'postgresql':
    sqlite_engine = create_engine('sqlite:///theatre.db')
    tables = ['plays', 'theatres', 'performances', 'customers', 'tickets']
    for table in tables:
        data = pd.read_sql_query(
            f"""
            SELECT
                *
            FROM {table};
            """,
            con=sqlite_engine,
        )
        data.to_sql(table, engine, if_exists='replace', index=False)
        print(table, ':', len(data), 'Zeilen')
    sqlite_engine.dispose()
else:
    print('SQLite-Backup: theatre.db')

### Backup mit SQLite

Falls PostgreSQL nicht erreichbar ist, führen Sie statt der Verbindungszelle diese beiden Zeilen aus und fahren danach mit der nächsten Zelle fort:

```python
# Verbindung zur lokalen SQLite-Datei herstellen.
db_connection_url = 'sqlite:///theatre.db'
engine = create_engine(db_connection_url)
```

Alle Aufgaben funktionieren unverändert mit beiden Datenbanken.

---

<div style="background-color:#2E4057; color:white; padding:16px 24px; border-radius:6px; font-size:18px; margin-top:8px;">
&#9654;&nbsp; <strong>Ab hier beginnt die bewertete Prüfung — 40 Punkte</strong><br>
<span style="font-size:15px; font-weight:normal;">Tragen Sie Ihre Antworten direkt in die vorgesehenen Zellen ein und führen Sie alle Code-Zellen aus.</span>
</div>

Die Aufgaben sind unabhängig voneinander lösbar. Verändern Sie die Tabellen in der Datenbank nicht; verwenden Sie eigene Variablen je Aufgabe. Teil B enthält bewusst fachlich fehlerhaften, aber lauffähigen Code.

**Schriftliche Antworten und Begründungen** tragen Sie jeweils in die Markdown-Zelle «Ihre Antwort / Begründung» unter der Aufgabe ein, Code in die zugehörige Codezelle.

---
## Teil A – Konzeptfragen (12 Punkte)

### A1 (4 Punkte)

Eine Kollegin schlägt vor, die Tabelle `customers` aufzulösen und `firstname`, `lastname`, `email` und `city` direkt in der Tabelle `tickets` zu speichern: *«Dann brauchen wir für Kundenauswertungen keinen JOIN mehr.»*

Ist das sinnvoll? Begründen Sie mit zwei konkreten Nachteilen und beziehen Sie sich auf die Daten in `theatre.db`.

**Ihre Antwort / Begründung:**

> [Hier eintragen]

### A2 (4 Punkte)

Beim Import neuer Verkäufe soll folgende Zeile in der Tabelle `tickets` gespeichert werden:

| ticketid | performanceid | customerid | seat_number | paid_price |
|---|---|---|---|---:|
| T02001 | A9999 | K0001 | A01 | 32.00 |

Die `ticketid` `T02001` kommt in `tickets` noch nicht vor. Eine Aufführung `A9999` gibt es in `performances` nicht. Ein Mitarbeiter meint: *«Kein Problem – die `ticketid` ist ja neu und eindeutig.»*

a) Hat er recht? Begründen Sie mit dem Schema von `theatre.db`.

b) Wie sollte der Fall fachlich korrekt gelöst werden? Kein SQL-Code verlangt.

**Ihre Antwort / Begründung:**

> [Hier eintragen]

### A3 (4 Punkte)

Nach einem Import zählt ein Mitarbeiter für eine Aufführung im «Theater am Park» **34 Tickets**. Laut `theatres.total_seats` hat das Theater **32 Sitzplätze**.

Ist das ein Datenfehler? Begründen Sie Ihre Antwort und beschreiben Sie kurz, wie man solche Fälle in `theatre.db` finden könnte. Kein SQL-Code verlangt.

**Ihre Antwort / Begründung:**

> [Hier eintragen]

---
## Teil B – Fehleranalyse (16 Punkte)

### B1 (8 Punkte)

**Ziel:** Die Anzahl Aufführungen je Theaterstück ausgeben – für **alle** Theaterstücke in der Datenbank.

Die Outputtabelle muss `playid`, `title` und `performance_count` enthalten: eine Zeile je Theaterstück, nach `playid` aufsteigend sortiert.

Die Tabelle `plays` enthält 15 Theaterstücke. Die Dramaturgin wundert sich über das Ergebnis: *«Wir haben im ganzen Zeitraum nur 90 Aufführungen gespielt – und ein Stück fehlt in der Liste.»*

**Aufgabe:** Schreiben Sie den korrigierten Code in die Codezelle. Begründen Sie jede Änderung.

**Zu korrigierende SQL-Abfrage:**
```sql
-- Geplante Aufführungen je Theaterstück auswerten.
SELECT
    s.playid,
    s.title,
    COUNT(a.performanceid) AS performance_count
FROM plays AS s
INNER JOIN performances AS a
    ON s.playid = a.playid
INNER JOIN tickets AS t
    ON a.performanceid = t.performanceid
GROUP BY
    s.playid,
    s.title
ORDER BY
    s.playid ASC;
```

**Ausgabe der oben gezeigten SQL-Abfrage vor der Korrektur:**

Diese Ausgabe zeigt den fehlerhaften Ausgangszustand, nicht das gesuchte korrekte Ergebnis.

```text
playid                 title  performance_count
  ST01 Die verschwundene Uhr                132
  ST02        Der letzte Zug                119
  ST03    Sommer im Innenhof                109
  ST04  Ein Platz am Fenster                145
  ST05     Die zweite Chance                112
  ST06      Das leere Zimmer                153
  ST07 Briefe aus dem Norden                 92
  ST08 Ein Sonntag im Garten                103
  ST09       Die falsche Tür                 85
  ST10    Hinter dem Vorhang                157
  ST11       Das grosse Fest                124
  ST12 Besuch um Mitternacht                149
  ST13       Der neue Anfang                343
  ST14     Eine kleine Reise                177
```

**Ihre Antwort / Begründung:**

> [Hier eintragen]

In [ ]:
# SQL-Abfrage formulieren.
query_B1 = """
-- Korrigierte SQL-Abfrage hier eintragen
"""

# Abfrage ausführen und Ergebnis als DataFrame laden.
df_B1 = pd.read_sql_query(query_B1, con=engine)

# Ergebnis ausgeben.
print(df_B1.to_string(index=False))

### B2 (8 Punkte)

**Ziel:** Den durchschnittlich bezahlten Ticketpreis je Theater ausgeben, über alle ausgegebenen Tickets.

Die Outputtabelle muss `name` und `avg_price` (CHF, zwei Dezimalstellen) enthalten: eine Zeile je Theater mit Tickets, nach `avg_price` aufsteigend sortiert.

Die Geschäftsleitung ist skeptisch: *«Bei unseren vielen Rabattaktionen haben wir pro Ticket nie so viel eingenommen.»*

**Aufgabe:** Schreiben Sie den korrigierten Code in die Codezelle. Begründen Sie jede Änderung.

**Zu korrigierende SQL-Abfrage:**
```sql
-- Durchschnittlichen Ticketpreis je Theater auswerten.
SELECT
    h.name,
    ROUND(CAST(AVG(a.base_price) AS NUMERIC), 2) AS avg_price
FROM theatres AS h
INNER JOIN performances AS a
    ON h.theatreid = a.theatreid
INNER JOIN tickets AS t
    ON a.performanceid = t.performanceid
WHERE
    t.paid_price > 0
GROUP BY
    h.theatreid,
    h.name
ORDER BY
    avg_price ASC;
```

**Ausgabe der oben gezeigten SQL-Abfrage vor der Korrektur:**

Diese Ausgabe zeigt den fehlerhaften Ausgangszustand, nicht das gesuchte korrekte Ergebnis.

```text
               name  avg_price
    Theater am Park      30.77
     Theater im Hof      32.46
     Bühne am Fluss      36.54
         Neue Bühne      40.90
Schauspielhaus West      44.66
```

**Ihre Antwort / Begründung:**

> [Hier eintragen]

In [ ]:
# SQL-Abfrage formulieren.
query_B2 = """
-- Korrigierte SQL-Abfrage hier eintragen
"""

# Abfrage ausführen und Ergebnis als DataFrame laden.
df_B2 = pd.read_sql_query(query_B2, con=engine)

# Ergebnis ausgeben.
print(df_B2.to_string(index=False))

---
## Teil C – SQL-Abfragen schreiben (12 Punkte)

### C1 (4 Punkte)

Für ein Familienwochenende sucht das Marketing alle Stücke der Genres `Komödie` und `Familie`, die **höchstens 90 Minuten** dauern.

Die Outputtabelle muss `title`, `genre` und `duration_min` enthalten: eine Zeile je Stück, nach `duration_min` aufsteigend sortiert.

Begründen Sie Ihre Filterbedingung.

**Ihre Antwort / Begründung:**

> [Hier eintragen]

In [ ]:
# SQL-Abfrage formulieren.
query_C1 = """
-- SQL-Abfrage hier eintragen
"""

# Abfrage ausführen und Ergebnis als DataFrame laden.
df_C1 = pd.read_sql_query(query_C1, con=engine)

# Ergebnis ausgeben.
print(df_C1.to_string(index=False))

### C2 (4 Punkte)

Wie hoch ist der durchschnittliche Rabatt pro Ticket je Genre (`genre`), über alle ausgegebenen Tickets einschliesslich Freikarten? Der Rabatt eines Tickets ist die Differenz zwischen dem Basispreis der Aufführung und dem bezahlten Preis.

Die Outputtabelle muss `genre` und `avg_discount` (CHF, gerundet auf zwei Dezimalstellen mit `ROUND(CAST(... AS NUMERIC), 2)`) enthalten: eine Zeile je Genre, nach `avg_discount` absteigend sortiert.

Begründen Sie, welche Tabellen Sie verbinden müssen und warum.

**Ihre Antwort / Begründung:**

> [Hier eintragen]

In [ ]:
# SQL-Abfrage formulieren.
query_C2 = """
-- SQL-Abfrage hier eintragen
"""

# Abfrage ausführen und Ergebnis als DataFrame laden.
df_C2 = pd.read_sql_query(query_C2, con=engine)

# Ergebnis ausgeben.
print(df_C2.to_string(index=False))

### C3 (4 Punkte)

Wie viel haben die einzelnen Kunden mit Wohnort **Winterthur** insgesamt für ihre Tickets bezahlt?

Die Outputtabelle muss `customerid`, `firstname`, `lastname` und `total_paid` (CHF, zwei Dezimalstellen) enthalten: eine Zeile je Kunde aus Winterthur mit mindestens einem Ticket, nach `total_paid` absteigend und bei gleichem Betrag nach `customerid` aufsteigend sortiert.

Begründen Sie, welche Tabellen Sie verbinden und welchen JOIN-Typ Sie verwenden.

**Ihre Antwort / Begründung:**

> [Hier eintragen]

In [ ]:
# SQL-Abfrage formulieren.
query_C3 = """
-- SQL-Abfrage hier eintragen
"""

# Abfrage ausführen und Ergebnis als DataFrame laden.
df_C3 = pd.read_sql_query(query_C3, con=engine)

# Ergebnis ausgeben.
print(df_C3.to_string(index=False))

---
### Notebook-Footer (vorgegeben)

In [ ]:
# Bibliotheken für die technischen Angaben laden.
import platform
from datetime import datetime

# Python-Version, Betriebssystem und Zeitpunkt ausgeben.
print('Python:', platform.python_version())
print('System:', platform.system())
print('Zeitpunkt:', datetime.now().isoformat(timespec='seconds'))